# Nettoyage, harmonisation et fusion des données des trois sites

## Installation et imports

In [1]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','pyspark','--quiet'])
print('OK')

OK



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, shutil, subprocess, time
import pandas as pd
import re as _re

subprocess.run('pkill -f SparkSubmit', shell=True, capture_output=True)
subprocess.run('pkill -f pyspark', shell=True, capture_output=True)
subprocess.run('pkill -f GatewayServer', shell=True, capture_output=True)
time.sleep(2)

try:
    from pyspark import SparkContext
    if SparkContext._active_spark_context:
        SparkContext._active_spark_context.stop()
    SparkContext._active_spark_context = None
    SparkContext._gateway = None
except Exception:
    pass

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, IntegerType
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName('Immobilier') \
    .config('spark.driver.memory', '2g') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.ui.enabled', 'false') \
    .config('spark.driver.maxResultSize', '1g') \
    .config('spark.memory.offHeap.enabled', 'true') \
    .config('spark.memory.offHeap.size', '512m') \
    .master('local[2]') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
DATA_DIR = os.getcwd()
print(f'Spark version : {spark.version}')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/12 11:04:48 WARN Utils: Your hostname, codespaces-59c905, resolves to a loopback address: 127.0.0.1; using 10.0.1.121 instead (on interface eth0)
26/04/12 11:04:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/12 11:04:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 4.1.1


## Nettoyage de la table PAP

In [4]:
# Chargement du fichier 
pap_raw_path = os.path.join(DATA_DIR, 'pap_raw.csv')
df_pap_brut = pd.read_csv(pap_raw_path, on_bad_lines='skip', sep=None, engine='python')

In [5]:
# Nettoyage des noms de colonnes 
df_pap_brut.columns = [c.strip().strip('\ufeff').strip('"') for c in df_pap_brut.columns]
print('Colonnes detectees :', list(df_pap_brut.columns))
print(f'Lignes brutes : {len(df_pap_brut)}')

Colonnes detectees : ['type', 'ville', 'prix', 'surface', 'nb_pieces', 'nb_chambres', 'description']
Lignes brutes : 1470


In [6]:
# Séparation du type en deux colonnes (transaction, type_bien)
df_pap_brut[['transaction', 'type_bien']] = df_pap_brut['type'].str.split('-', n=1, expand=True)
df_pap_brut.drop(columns=['type'], inplace=True)

In [7]:
# Ajout de la source
df_pap_brut['source'] = 'pap'

In [8]:
# Extraction de la ville et du code postal 
def extract_ville_cp(s):
    if pd.isna(s):
        return pd.NA, pd.NA
    s = s.replace('\xa0', ' ')
    cp_match = _re.search(r'\((\d{5})\)', s)
    cp = cp_match.group(1) if cp_match else pd.NA
    ville_match = _re.search(r'([\w-]+)\s+([\w-]+)\s*\(\d{5}\)', s)
    if ville_match:
        avant_dernier = ville_match.group(1)
        dernier       = ville_match.group(2)
        ville = avant_dernier if _re.match(r'^\d+\w*$', dernier) else dernier
    else:
        m = _re.search(r'([\w-]+)\s*\(\d{5}\)', s)
        ville = m.group(1) if m else pd.NA
    return ville, cp

df_pap_brut[['ville_clean', 'code_postal']] = df_pap_brut['ville'].apply(
    lambda s: pd.Series(extract_ville_cp(s))
)
df_pap_brut.drop(columns=['ville'], inplace=True)
df_pap_brut.rename(columns={'ville_clean': 'ville'}, inplace=True)

In [9]:
# Conversion des types 
df_pap_brut['surface'] = pd.to_numeric(df_pap_brut['surface'], errors='coerce')
for col in ['nb_pieces', 'nb_chambres', 'prix']:
    df_pap_brut[col] = pd.to_numeric(df_pap_brut[col], errors='coerce').astype('Int64')

In [10]:
# Réordonner les colonnes
cols_ordre = ['transaction', 'type_bien', 'prix', 'surface',
              'nb_pieces', 'nb_chambres', 'ville', 'code_postal', 'description', 'source']
cols_ordre = [c for c in cols_ordre if c in df_pap_brut.columns]
df_pap_brut = df_pap_brut[cols_ordre]

print(f'Lignes après nettoyage : {len(df_pap_brut)}')
print(df_pap_brut.dtypes)
print()
print(df_pap_brut.head(5).to_string())

Lignes après nettoyage : 1470
transaction     object
type_bien       object
prix             Int64
surface        float64
nb_pieces        Int64
nb_chambres      Int64
ville           object
code_postal     object
description     object
source          object
dtype: object

  transaction     type_bien    prix  surface  nb_pieces  nb_chambres  ville code_postal                                                                                                                                                                                                description source
0      ventes  appartements  610000    65.00          3            2  Paris       75010   3 pièces de 65 m², au 154 rue La Fayette  au croisement entre le rue La Fayette et la rue du Faubourg St-Denis,  2 min du métro Gare du Nord.  - 5eme étage avec ascenseur, appartement très lumineux...    pap
1      ventes  appartements  532000    65.00          3            2  Paris       75013  Paris 13 ème – Quartier Jeanne d'Arc.  D

In [11]:
# Export du dataframe
pap_clean_path = os.path.join(DATA_DIR, 'pap_clean.csv')
df_pap_brut.to_csv(pap_clean_path, sep=';', index=False, encoding='utf-8-sig', quoting=1)
print(f'\nPAP_clean.csv exporté : {len(df_pap_brut)} lignes')


PAP_clean.csv exporté : 1470 lignes


## Chargement des trois tables

In [12]:
# PAP
df_pap_raw = spark.read \
    .option('header', 'true') \
    .option('sep', ';') \
    .option('encoding', 'UTF-8') \
    .csv(os.path.join(DATA_DIR, 'pap_clean.csv'))
print(f'PAP : {df_pap_raw.count()} lignes')

# Orpi
df_orpi_raw = spark.read \
    .option('multiline', 'true') \
    .json(os.path.join(DATA_DIR, 'orpi_raw.json'))
print(f'Orpi : {df_orpi_raw.count()} lignes')

# Entreparticuliers
df_ep_raw = spark.read \
    .option('multiline', 'true') \
    .json(os.path.join(DATA_DIR, 'entreparticuliers_raw.json'))
print(f'EP : {df_ep_raw.count()} lignes')

PAP : 1470 lignes
Orpi : 544 lignes
EP : 480 lignes


## Harmonisation et fusion des tables

In [13]:
# PAP
df_pap = df_pap_raw \
    .withColumnRenamed('transaction', 'type_transaction') \
    .withColumn('type_transaction',
        F.when(F.col('type_transaction').isin('ventes','vente'), 'vente')
         .when(F.col('type_transaction').isin('locations','location'), 'location')
         .otherwise(F.col('type_transaction'))) \
    .withColumn('type_bien',
        F.when(F.lower('type_bien').isin('appartements','appartement'), 'Appartement')
         .when(F.lower('type_bien').isin('maisons','maison'), 'Maison')
         .when(F.lower('type_bien').isin('studios','studio'), 'Studio')
         .when(F.lower('type_bien').isin('villas','villa'), 'Villa')
         .otherwise(F.initcap('type_bien'))) \
    .withColumn('prix',        F.col('prix').cast(DoubleType())) \
    .withColumn('surface',     F.col('surface').cast(DoubleType())) \
    .withColumn('nb_pieces',   F.col('nb_pieces').cast(DoubleType())) \
    .withColumn('nb_chambres', F.col('nb_chambres').cast(DoubleType())) \
    .withColumn('source',      F.lit('pap')) \
    .select('type_transaction','type_bien','prix','surface',
            'nb_pieces','nb_chambres','ville','code_postal','description','source')
print(f'PAP harmonisé : {df_pap.count()} lignes')

# Orpi
df_orpi = df_orpi_raw \
    .withColumn('prix',        F.col('prix').cast(DoubleType())) \
    .withColumn('surface',     F.col('surface').cast(DoubleType())) \
    .withColumn('nb_pieces',   F.col('nb_pieces').cast(DoubleType())) \
    .withColumn('nb_chambres', F.lit(None).cast(DoubleType())) \
    .select('type_transaction','type_bien','prix','surface',
            'nb_pieces','nb_chambres','ville','code_postal','description','source')
print(f'Orpi harmonisé : {df_orpi.count()} lignes')

# Entreparticuliers
df_ep = df_ep_raw \
    .withColumn('prix',        F.col('prix').cast(DoubleType())) \
    .withColumn('surface',     F.col('surface').cast(DoubleType())) \
    .withColumn('nb_pieces',   F.col('nb_pieces').cast(DoubleType())) \
    .withColumn('nb_chambres', F.lit(None).cast(DoubleType())) \
    .withColumn('description',
        F.when(F.col('description') == '', None).otherwise(F.col('description'))) \
    .select('type_transaction','type_bien','prix','surface',
            'nb_pieces','nb_chambres','ville','code_postal','description','source')
print(f'EP harmonisé : {df_ep.count()} lignes')

# Fusion
df = df_pap.union(df_orpi).union(df_ep)
print(f'Total fusionné : {df.count()} lignes')
df.printSchema()

PAP harmonisé : 1470 lignes
Orpi harmonisé : 544 lignes
EP harmonisé : 480 lignes
Total fusionné : 2494 lignes
root
 |-- type_transaction: string (nullable = true)
 |-- type_bien: string (nullable = true)
 |-- prix: double (nullable = true)
 |-- surface: double (nullable = true)
 |-- nb_pieces: double (nullable = true)
 |-- nb_chambres: double (nullable = true)
 |-- ville: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- description: string (nullable = true)
 |-- source: string (nullable = true)



In [14]:
# Aperçu du dataset fusionné
print(f'Dataset fusionné : {df.count()} lignes')
df.groupBy('source','type_transaction').count().orderBy('source').show()
df.show(5, truncate=60)

Dataset fusionné : 2494 lignes
+-----------------+----------------+-----+
|           source|type_transaction|count|
+-----------------+----------------+-----+
|entreparticuliers|        location|  378|
|entreparticuliers|           vente|  102|
|             orpi|           vente|  269|
|             orpi|        location|  275|
|              pap|           vente|  726|
|              pap|        location|  744|
+-----------------+----------------+-----+



+----------------+-----------+--------+-------+---------+-----------+-----+-----------+------------------------------------------------------------+------+
|type_transaction|  type_bien|    prix|surface|nb_pieces|nb_chambres|ville|code_postal|                                                 description|source|
+----------------+-----------+--------+-------+---------+-----------+-----+-----------+------------------------------------------------------------+------+
|           vente|Appartement|610000.0|   65.0|      3.0|        2.0|Paris|      75010|3 pièces de 65 m², au 154 rue La Fayette  au croisement e...|   pap|
|           vente|Appartement|532000.0|   65.0|      3.0|        2.0|Paris|      75013|Paris 13 ème – Quartier Jeanne d'Arc.  Dans une résidence...|   pap|
|           vente|Appartement|128000.0|   9.31|      1.0|        1.0|Paris|      75005|Chambre Métro Maubert-Mutualité.  Au 6ᵉ étage d'un immeub...|   pap|
|           vente|Appartement|330000.0|   28.0|      2.0|       

## Export du dataframe fusionné

In [15]:
out = os.path.join(DATA_DIR, 'immobilier_fusionne.csv')
if os.path.exists(out):
    shutil.rmtree(out) if os.path.isdir(out) else os.remove(out)
df.toPandas().to_csv(out, index=False, encoding='utf-8')
print(f'immobilier_fusionne.csv exporté : {df.count()} lignes')

immobilier_fusionne.csv exporté : 2494 lignes


In [16]:
spark.stop()
print('Spark terminé.')

Spark terminé.
